# Week 5 Coding Practice Solution: Predicting Outcomes and Finding Hidden Structure

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/obscrivn/DataScience-book/blob/main/module05/week5_trees_factor_analysis_practice_solution.ipynb)

This companion provides worked code and concise model responses for the Week 5 coding practice. The interpretation examples show one defensible reading of this fixed instructional split; students may use different wording when their claims remain supported by the evidence.

## Part A solution — Trees for prediction

The target is the quantitative disease-progression measure observed one year after baseline. The ten baseline measurements are predictors. `bmi` is a plausible candidate for an early split because it is a baseline health measurement that may differentiate outcome levels, but that expectation must be checked in the fitted tree. Holding out test rows matters because a model can fit patterns specific to its training rows without predicting new people well.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.datasets import load_diabetes, load_wine
from sklearn.decomposition import FactorAnalysis
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor, plot_tree

sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 42

diabetes = load_diabetes(as_frame=True)
tree_data = diabetes.data.copy()
tree_data["progression"] = diabetes.target
X = tree_data.drop(columns="progression")
y = tree_data["progression"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE
)

small_tree = DecisionTreeRegressor(
    max_depth=3, min_samples_leaf=12, random_state=RANDOM_STATE
).fit(X_train, y_train)

print(f"Training rows: {len(X_train)} | Test rows: {len(X_test)}")
display(tree_data.head())

In [ ]:
fig, ax = plt.subplots(figsize=(17, 8))
plot_tree(
    small_tree, feature_names=X.columns, filled=True, rounded=True,
    precision=1, fontsize=9, ax=ax
)
ax.set_title("Small regression tree: predicted one-year progression")
plt.show()

print(f"Depth: {small_tree.get_depth()} | Leaves: {small_tree.get_n_leaves()}")

### Interpreting one path

One leftmost path says: baseline `bmi` is at or below about 0.0, `s5` is at or below about 0.0, and the same `s5` condition is met at the next split. The leaf predicts a progression value of about 80.3 for the 45 training observations that reach it. This is an average of a training group, not an exact or guaranteed value for every future person with those conditions.

In [ ]:
deep_tree = DecisionTreeRegressor(random_state=RANDOM_STATE)
forest = RandomForestRegressor(
    n_estimators=300, min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=-1
)
models = {
    "Small tree (depth 3)": small_tree,
    "Deep tree": deep_tree,
    "Random forest": forest,
}

def regression_scores(model, X_values, y_values):
    predictions = model.predict(X_values)
    return {
        "MAE": mean_absolute_error(y_values, predictions),
        "RMSE": mean_squared_error(y_values, predictions) ** 0.5,
    }

score_rows = []
for name, model in models.items():
    if name != "Small tree (depth 3)":
        model.fit(X_train, y_train)
    train_scores = regression_scores(model, X_train, y_train)
    test_scores = regression_scores(model, X_test, y_test)
    score_rows.append({
        "model": name,
        "train_MAE": train_scores["MAE"],
        "test_MAE": test_scores["MAE"],
        "train_RMSE": train_scores["RMSE"],
        "test_RMSE": test_scores["RMSE"],
    })

score_table = pd.DataFrame(score_rows).set_index("model").round(1)
display(score_table)

fig, ax = plt.subplots(figsize=(8, 4.5))
score_table[["train_RMSE", "test_RMSE"]].plot.bar(
    ax=ax, color=["#56B4E9", "#D55E00"]
)
ax.set_ylabel("RMSE (lower is better)")
ax.set_xlabel("Model")
ax.set_title("Training versus unseen-test error")
ax.tick_params(axis="x", rotation=0)
ax.legend(["Training", "Test"], title="Rows used for scoring")
plt.tight_layout()
plt.show()

print(f"Deep tree depth: {deep_tree.get_depth()} | leaves: {deep_tree.get_n_leaves()}")

### Complexity diagnosis

The deep tree has the lowest training RMSE—0.0—because it fits the training rows perfectly. That is not enough to choose it: its test RMSE is 77.1, much worse than the small tree's 56.3 and the random forest's 53.1. The zero-versus-77.1 gap is strong evidence of overfitting for this split.

The random forest is the strongest predictor on these held-out rows, but the small tree may still be preferable when a stakeholder needs a compact, auditable rule and its modestly higher error is acceptable. A decision should also consider the cost of an error, stability across additional splits, fairness, and whether the data represent the intended population.

In [ ]:
importance_table = pd.DataFrame(
    {
        "small_tree": small_tree.feature_importances_,
        "random_forest": forest.feature_importances_,
    },
    index=X.columns,
).sort_values("random_forest", ascending=True)

display(importance_table.sort_values("random_forest", ascending=False).round(3))

fig, ax = plt.subplots(figsize=(8, 5))
importance_table.plot.barh(ax=ax, color=["#0072B2", "#009E73"])
ax.set_xlabel("Model feature importance")
ax.set_ylabel("Baseline measurement")
ax.set_title("Importance can differ between one tree and many trees")
ax.legend(["Small tree", "Random forest"])
plt.tight_layout()
plt.show()

### Feature-importance correction

A high importance means the fitted model used that measurement to reduce prediction error in this dataset. It does not show that changing the measurement would change progression, nor does it establish that less-important variables are irrelevant in every population. A causal claim would need a study design that addresses confounding—such as a well-designed experiment or a defensible causal observational design—and domain expertise about the measurement.

### One meaningful tree change

Here we increase the tree depth from 3 to 5 while retaining `min_samples_leaf=12`. We expect training error to decrease because the model has more opportunities to split. Test error may improve if the extra splits capture stable structure, or worsen if they fit idiosyncrasies of the training rows.

In [ ]:
depth_five_tree = DecisionTreeRegressor(
    max_depth=5, min_samples_leaf=12, random_state=RANDOM_STATE
).fit(X_train, y_train)

transfer_scores = pd.DataFrame(
    {
        "train": regression_scores(depth_five_tree, X_train, y_train),
        "test": regression_scores(depth_five_tree, X_test, y_test),
    }
).T.round(1)
display(transfer_scores)
print(
    f"Depth: {depth_five_tree.get_depth()} | Leaves: {depth_five_tree.get_n_leaves()}"
)

The depth-5 tree lowers training RMSE to 48.8 but increases test RMSE to 57.9, compared with 54.7/56.3 for the depth-3 tree. On this split, the extra complexity improves fit to training data without improving prediction on unseen data. This is evidence against selecting the deeper version automatically.

## Part B solution — Finding possible latent structure

This part uses the wine measurements without the cultivar label. The goal is not prediction; it is to inspect whether related observed variables might reflect a smaller number of shared patterns.

In [ ]:
wine = load_wine(as_frame=True)
wine_measurements = wine.data.copy()
correlations = wine_measurements.corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(correlations, cmap="vlag", center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title("Relationships among observed wine measurements")
plt.tight_layout()
plt.show()

wine_scaled = StandardScaler().fit_transform(wine_measurements)
factor_model = FactorAnalysis(n_components=2, random_state=RANDOM_STATE).fit(wine_scaled)
loadings = pd.DataFrame(
    factor_model.components_.T,
    index=wine_measurements.columns,
    columns=["Factor 1", "Factor 2"],
)

display(loadings.reindex(loadings.abs().max(axis=1).sort_values(ascending=False).index).round(2))

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(loadings, annot=True, fmt=".2f", cmap="vlag", center=0, ax=ax)
ax.set_title("Two-factor loading pattern (standardized measurements)")
plt.tight_layout()
plt.show()

### Interpreting the loading pattern

The correlation heatmap suggests that `total_phenols`, `flavanoids`, and `od280/od315_of_diluted_wines` are positively related, so investigating shared structure is reasonable. In the two-factor output, these variables have large absolute loadings on Factor 1. A tentative label might be **phenolic composition pattern**. `alcohol` and `color_intensity` have large positive loadings on Factor 2, so a tentative label might be **alcohol/color-intensity pattern**.

Those names are interpretations supplied by an analyst, not facts automatically named by factor analysis. The signs can also be reversed without changing the fitted factor. Additional subject-matter evidence, replication, and a deliberate choice of model structure would be needed before treating either label as a validated construct.

# Final synthesis solution

To predict whether a customer will renew a subscription, use a supervised predictive model such as a decision tree or random forest, trained with a known renewal outcome and evaluated on held-out customers. To investigate whether ten customer measurements reflect a smaller number of underlying characteristics, use factor analysis because the question concerns shared structure among observed variables rather than predicting a target.

A defensible final check is: confirm the target matches the prediction question; evaluate predictive claims on unseen rows; avoid choosing complexity from training error alone; treat importance as model evidence rather than causation; label factors tentatively from their loadings; and keep prediction distinct from explanation or causal proof.